# 02 — Classical Rheological Modelling: Power Law vs. Herschel–Bulkley

## Theoretical Formulations

### 1. Ostwald–de Waele (Power Law) Model
$$\tau = K \dot{\gamma}^n$$
$$\eta_a = K \dot{\gamma}^{n-1}$$
- $K$: Consistency index ($\text{Pa}\cdot\text{s}^n$)
- $n$: Flow behaviour index (dimensionless; $n < 1$ for shear-thinning pseudoplastics)

### 2. Herschel–Bulkley Model
$$\tau = \tau_y + K \dot{\gamma}^n$$
$$\eta_a = \frac{\tau_y}{\dot{\gamma}} + K \dot{\gamma}^{n-1}$$
- $\tau_y$: Yield stress ($\text{Pa}$); fluid requires a threshold stress before undergoing continuous deformation.

### Objectives
1. Estimate parameters via nonlinear least-squares regression with physical constraints ($K>0, n>0, \tau_y \ge 0$).
2. Evaluate goodness of fit ($R^2$, RMSE, MAE, log-RMSE).
3. Perform evidence-based model selection without assuming yield stress for all fluids.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, TABLES_DIR
from src.data_processing import load_dataset
from src.physics_models import (
    fit_all_conditions, fit_power_law, fit_herschel_bulkley,
    power_law_stress, herschel_bulkley_stress, select_best_model
)
from src.visualization import plot_model_fit

df = load_dataset(PROCESSED_DATA_DIR / 'processed_data.csv')
print(f'Loaded {len(df)} preprocessed measurements.')

## 1. Single-Condition Classical Fitting Demo
Fitting a sample condition to compare Power Law and Herschel–Bulkley fits directly.

In [ ]:
sub = df[(df['fluid_type'] == 'XG') & (df['concentration'] == 1.0) & (df['temperature'] == 25)]
sr = sub['shear_rate'].values
ss = sub['shear_stress'].values

pl_fit = fit_power_law(sr, ss, condition_label='XG 1.0wt% 25C')
hb_fit = fit_herschel_bulkley(sr, ss, condition_label='XG 1.0wt% 25C')
best_fit, reason = select_best_model(pl_fit, hb_fit)

print(pl_fit.summary_str())
print(hb_fit.summary_str())
print(f'Selected Best Model: {best_fit.model_name} -> {reason}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Power Law
axes[0].loglog(sr, ss, 'ko', label='Experimental Data')
axes[0].loglog(sr, power_law_stress(sr, pl_fit.parameters['K'], pl_fit.parameters['n']), 'r-', label=f'Power Law (R²={pl_fit.metrics["R2"]:.4f})')
axes[0].set_xlabel('Shear Rate (s⁻¹)')
axes[0].set_ylabel('Shear Stress (Pa)')
axes[0].set_title('Power Law Fit')
axes[0].grid(True, which='both', ls='--', alpha=0.3)
axes[0].legend()

# Herschel-Bulkley
axes[1].loglog(sr, ss, 'ko', label='Experimental Data')
axes[1].loglog(sr, herschel_bulkley_stress(sr, hb_fit.parameters['tau_y'], hb_fit.parameters['K'], hb_fit.parameters['n']), 'b-', label=f'Herschel-Bulkley (R²={hb_fit.metrics["R2"]:.4f})')
axes[1].set_xlabel('Shear Rate (s⁻¹)')
axes[1].set_ylabel('Shear Stress (Pa)')
axes[1].set_title('Herschel–Bulkley Fit')
axes[1].grid(True, which='both', ls='--', alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. Fitting Across All Experimental Conditions
Automated parameter estimation and model selection across every combination of fluid, concentration, and temperature.

In [ ]:
fit_results = fit_all_conditions(df, mode='stress')
fit_table = pd.read_csv(TABLES_DIR / 'classical_model_fits.csv')
display(fit_table[['condition', 'best_model', 'PL_K', 'PL_n', 'PL_R2', 'HB_tau_y', 'HB_K', 'HB_n', 'HB_R2', 'selection_reason']])

### Rheological Insights & Model Selection Analysis
- **Carboxymethyl Cellulose (CMC)**: Across 0.5%–1.5% concentrations, fitted yield stress values $\tau_y$ are either negligible ($< 10^{-4}\text{ Pa}$) or statistically indistinguishable from zero ($p > 0.05$). The Power Law model is chosen by Occam's razor as the more parsimonious model.
- **Xanthan Gum (XG)**: At higher concentrations (0.5% and 1.0%), XG forms rigid intermolecular double-stranded helix networks with significant yield stresses ($\tau_y \approx 0.15 - 1.2\text{ Pa}$), where the Herschel–Bulkley model provides superior low-shear accuracy.